# Phase 4: Lifetime Value (LTV) Prediction
## Customer Churn & LTV Prediction Engine

This notebook focuses on estimating **Customer Lifetime Value (LTV)**. It explores methods to model LTV: regression approaches predicting continuous value outputs (`TotalCharges`), and statistical models like BG/NBD and Gamma-Gamma (if using transactional cohort datasets) or customer cohort survival modeling.

### Objectives:
1. Define/derive LTV proxies using charges and contract lengths.
2. Train regression models to estimate total spend values.
3. Align churn probability and total spend expectations to calculate risk-adjusted LTV.
4. Conduct segmentation to categorize users into High, Medium, and Low-value tiers.

In [1]:
import pandas as pd
import numpy as np
import joblib
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# 1. Load the raw split datasets
X_train_raw = pd.read_csv('../data/processed/X_train.csv')
X_test_raw = pd.read_csv('../data/processed/X_test.csv')

# The target variable for our LTV regression model is 'TotalCharges'
y_train_ltv = X_train_raw['TotalCharges']
y_test_ltv = X_test_raw['TotalCharges']

# 2. DROP target-leaking columns from our training features
drop_cols = ['TotalCharges', 'charge_ratio', 'charges_difference']
X_train_ltv = X_train_raw.drop(columns=drop_cols)
X_test_ltv = X_test_raw.drop(columns=drop_cols)

# 3. Define the features for our new preprocessor
numeric_cols_ltv = ['tenure', 'MonthlyCharges', 'total_services']
categorical_cols_ltv = [col for col in X_train_ltv.columns if col not in numeric_cols_ltv]

# 4. Rebuild the pipeline for the regression task
num_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(drop='first', sparse_output=False))
])

preprocessor_ltv = ColumnTransformer(
    transformers=[
        ('num', num_pipeline, numeric_cols_ltv),
        ('cat', cat_pipeline, categorical_cols_ltv)
    ],
    remainder='drop'
)

# 5. Fit and transform the features
X_train_trans_ltv = preprocessor_ltv.fit_transform(X_train_ltv)
X_test_trans_ltv = preprocessor_ltv.transform(X_test_ltv)

print("LTV features prepared successfully!")
print("New training features shape:", X_train_trans_ltv.shape)


LTV features prepared successfully!
New training features shape: (5634, 35)


In [2]:
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1. Initialize Ridge Regression
ridge_reg = Ridge(alpha=1.0, random_state=42)

# 2. Fit the model on the training data
print("Training Ridge Regression for LTV...")
ridge_reg.fit(X_train_trans_ltv, y_train_ltv)

# 3. Predict on the test data
y_pred_ltv = ridge_reg.predict(X_test_trans_ltv)

# 4. Calculate metrics
mae = mean_absolute_error(y_test_ltv, y_pred_ltv)
rmse = np.sqrt(mean_squared_error(y_test_ltv, y_pred_ltv))
r2 = r2_score(y_test_ltv, y_pred_ltv)

print("\n=== LTV REGRESSION EVALUATION (RIDGE) ===")
print(f"Mean Absolute Error (MAE): ${mae:.2f}")
print(f"Root Mean Squared Error (RMSE): ${rmse:.2f}")
print(f"R-squared (R2 Score): {r2:.4f}")


Training Ridge Regression for LTV...

=== LTV REGRESSION EVALUATION (RIDGE) ===
Mean Absolute Error (MAE): $539.52
Root Mean Squared Error (RMSE): $689.26
R-squared (R2 Score): 0.9031


In [3]:
# 1. Load Churn model and its preprocessor
churn_model = joblib.load('../models/churn_model.joblib')
churn_preprocessor = joblib.load('../models/preprocessor.joblib')

# 2. Get Churn probabilities on test set
# We must use the original raw X_test because the Churn preprocessor expect those exact columns
X_test_trans_churn = churn_preprocessor.transform(X_test_raw)
churn_probs = churn_model.predict_proba(X_test_trans_churn)[:, 1]

# 3. Create a summary DataFrame for the test set
test_summary = pd.DataFrame({
    'Actual_Churn': y_test_ltv.index.map(df['Churn']) if 'df' in locals() else y_test_ltv * 0, # backup mapping
    'Actual_TotalCharges': y_test_ltv,
    'Predicted_TotalCharges': y_pred_ltv,
    'Churn_Probability': churn_probs
})

# Re-align actual churn labels from the indices
test_summary['Actual_Churn'] = X_test_raw.index.map(pd.read_csv('../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv')['Churn'].map({'Yes': 1, 'No': 0}))

# 4. Calculate Risk-Adjusted LTV
test_summary['Risk_Adjusted_LTV'] = test_summary['Predicted_TotalCharges'] * (1 - test_summary['Churn_Probability'])

# 5. Segment customers based on Median Spend and Churn Risk (threshold = 0.5)
median_spend = test_summary['Predicted_TotalCharges'].median()

def segment_customer(row):
    is_high_spend = row['Predicted_TotalCharges'] >= median_spend
    is_high_risk = row['Churn_Probability'] >= 0.5
    
    if is_high_spend and is_high_risk:
        return 'Rescue Target (High Spend, High Churn Risk)'
    elif is_high_spend and not is_high_risk:
        return 'Loyal Champion (High Spend, Low Churn Risk)'
    elif not is_high_spend and is_high_risk:
        return 'Low Priority Churn (Low Spend, High Churn Risk)'
    else:
        return 'Stable Core (Low Spend, Low Churn Risk)'

test_summary['Segment'] = test_summary.apply(segment_customer, axis=1)

# Print the segment distribution and average values
print("=== CUSTOMER SEGMENTATION SUMMARY ===")
print(test_summary['Segment'].value_counts())
print("\n=== AVERAGE VALUES BY SEGMENT ===")
print(test_summary.groupby('Segment')[['Predicted_TotalCharges', 'Churn_Probability', 'Risk_Adjusted_LTV']].mean())


=== CUSTOMER SEGMENTATION SUMMARY ===
Segment
Loyal Champion (High Spend, Low Churn Risk)        528
Low Priority Churn (Low Spend, High Churn Risk)    363
Stable Core (Low Spend, Low Churn Risk)            341
Rescue Target (High Spend, High Churn Risk)        177
Name: count, dtype: int64

=== AVERAGE VALUES BY SEGMENT ===
                                                 Predicted_TotalCharges  \
Segment                                                                   
Low Priority Churn (Low Spend, High Churn Risk)              484.296370   
Loyal Champion (High Spend, Low Churn Risk)                 4293.458739   
Rescue Target (High Spend, High Churn Risk)                 3010.439691   
Stable Core (Low Spend, Low Churn Risk)                      459.150796   

                                                 Churn_Probability  \
Segment                                                              
Low Priority Churn (Low Spend, High Churn Risk)           0.744923   
Loyal Champi